In [ ]:
import torch
import torch.optim as optim
from torchviz import make_dot
import pandas as pd
from itables import init_notebook_mode, show
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

init_notebook_mode(all_interactive=True)

## AACBR

In [ ]:
# Deep Arguing?
class GradualAACBR(torch.nn.Module):

    def __init__(self, nodes, A):
        super(GradualAACBR, self).__init__()
        # n = no.nodes
        # f = no.features
        self.nodes = nodes # (n, f) - list of nodes 
        self.A = A # (n, n) - Weighted adjacency matrix
        self.W = torch.nn.Parameter(torch.rand((nodes.shape[1]))) # Feature Weights (f)
    
    def compute_base_scores(self):
        # TODO: Base Scores must be between 0 and 1 - using sigmoid for now 
        return torch.sigmoid(torch.matmul(self.nodes, self.W))


    def forward(self, new_case, new_case_attacks, max_iters = 5):

        # TODO: Batch input of new_case attacks and find a way to do computations in parallel
        # Maybe duplicate base_scores into a B x n matrix where B is the size of the batch
        # and each row is just a copy of the base scores?
        # Then just update the base_scores for the batch corresponding to the new_case


        # TODO: new_case should do one of:
        #   CURRENT: set base scores of any that it attacks to 0 -> i.e. irrelevant arguments have no strength 
        #   POSSIBLE: compute it's base strength and use that + attacks to influence what it attacks 


        base_scores = self.compute_base_scores() # (n)

        # base_scores[new_case_attacks] * 0.000001 # TODO: Change how this handled

        base_scores = torch.where(new_case_attacks, 0.00001, base_scores)

        strengths = [base_scores]
        base_score_influence = torch.log(base_scores/(1-base_scores))
        
        # TODO: change to use one of the following stop conditions:
        #   (convergence under some epsilon or max iters reached) OR 
        #   sort the nodes topologically and figure out how to do a single pass with matrix operations -> Only works for ACYCLIC graphs 
        for i in range(max_iters):
            aggregations = torch.matmul(self.A.T, strengths[i]) # (incoming edges linear combination with strengths of node) -> n 
            influences = torch.sigmoid(base_score_influence  + aggregations)
            strengths.append(influences)

        return strengths 





In [ ]:
"""
C_3
 |
C_1  C_2
  \  |
    C_0
    
Each case has 10 features
"""
torch.manual_seed(42)

nodes = torch.rand((4, 5))
nodes[0] = nodes[0] * 0 # Default case has values set to 0 so base_score is computed as 0.5
A = torch.tensor([
                #     0   1   2   3
                    [ 0,  0,  0,  0], # 0
                    [-1,  0,  0,  0], # 1
                    [-1,  0,  0,  0], # 2
                    [ 0, -1,  0,  0], # 3
                ], dtype=torch.float32)

model = GradualAACBR(nodes, A)

result = model(new_case = torch.rand(1, 5), new_case_attacks=torch.tensor([False, False, True, False]), max_iters=3)

for r in result:
    print(r)

make_dot(r[-1], params=dict(model.named_parameters())).render("attached", format="png")

## Data Set

In [ ]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
connectionist_bench_sonar_mines_vs_rocks = fetch_ucirepo(id=151) 
  
# data (as pandas dataframes) 
X = connectionist_bench_sonar_mines_vs_rocks.data.features 
y = connectionist_bench_sonar_mines_vs_rocks.data.targets 

show(X)
print(y.nunique())



In [ ]:

encoder = LabelEncoder()
encoder.fit(y)
y = encoder.transform(y)


In [ ]:
print(encoder.classes_)
print(y)
print(type(y))

In [ ]:
X = X.values
print(X)

## Train Model

In [ ]:

# Once W is learned, create a new graph with training + validation set and test performance on the test set 


### Split into Training, Validation and Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Test Size:  {len(X_test)}")
print(f"Total Train Size:  {len(X_train)}")

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
print(f"Train Size:  {len(X_train)}")
print(f"Validation Size:  {len(X_val)}")



In [ ]:
print(X_train)

### Build AF

In [ ]:
# TODO: Use more sophisticated orders:

# Compare against the average for each column
means = X_train.mean(axis=0)
std = X_train.std(axis=0)

def gAVGCol(a, b):
    a = a - means
    b = b - means

    return all(np.sign(a) == np.sign(b)) and all(np.abs(a) > np.abs(b))

# Average across all columns:
def gAVG(a, b):
    return a.mean() > b.mean()
    

def gAVGSTD(a, b):
    if isinstance(a, torch.Tensor):
        a = a.numpy()
    if isinstance(b, torch.Tensor):
        b = b.numpy()

    if np.all(b == 0):
        return True
    if np.all(a == 0):
        return False
    a = np.where(np.abs(a - means) <= 2*std, 0, 1)
    b = np.where(np.abs(b - means) <= 2*std, 0, 1)

    return np.all(a & b == b) and not np.all(a & b == a)


In [ ]:
comparison_func = gAVGSTD

In [ ]:
DEFAULT_OUTCOME = 1

# Add default case (vector of 0s, with default outcome)
X_train = np.append(X_train, [[0]*X_train.shape[1]], axis=0)
y_train = np.append(y_train, [DEFAULT_OUTCOME])
print(X_train[-1])
print(y_train[-1])

In [ ]:
## TODO: Use more efficient implementation that sorts topologically as in AA-CBR Library

# +1 for default case
A = np.zeros((len(X_train), len(X_train)))


for i, a in enumerate(X_train):
    for j, b in enumerate(X_train):
        if y_train[i] == y_train[j]:
            continue
        # if comparison_func(a, b):
        #     print(i, j)
        #     concision = [y_train[k] == y_train[i] and comparison_func(a, c) and comparison_func(c, b) for k, c in enumerate(X_train)]
        #     print(np.where(concision))

        if comparison_func(a, b) and not any([y_train[k] == y_train[i] and comparison_func(a, c) and comparison_func(c, b) for k, c in enumerate(X_train)]):
            A[i, j] = -1
        elif all(a == b):
            A[i, j] = -1


In [ ]:
print(np.any(A == -1))

In [ ]:
print(A[:, -1])

In [ ]:

def show_graph_with_labels(adjacency_matrix):
    rows, cols = np.where(adjacency_matrix != 0)
    edges = zip(rows.tolist(), cols.tolist())
    gr = nx.Graph()
    gr.add_edges_from(edges)
    pos = nx.spiral_layout(gr)
    nx.draw(gr, pos, node_size=100)
    plt.show()

show_graph_with_labels(A)

In [ ]:
# TODO: Way to make this more efficient - rather than iterating through X_train
def get_new_case_attacks_mask(new_case, X_train):
    attacks = []
    for i, a in enumerate(X_train):
        attacks.append(not comparison_func(new_case, a))
    return attacks

### Train Model

In [ ]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32)
A_tensor = torch.tensor(A, dtype=torch.float32)


In [ ]:
model = GradualAACBR(X_train_tensor, A_tensor)
criterion = torch.nn.BCEWithLogitsLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

In [ ]:
print(model.W)

In [ ]:
for epoch in range(200):  # loop over the dataset multiple times

    running_loss = 0.0
    for i, input in enumerate(X_val_tensor, 0):
        # inputs, labels = data
        labels = y_val_tensor[i:i+1]

        # zero the parameter gradients
        optimizer.zero_grad()

        new_case_attacks = torch.tensor(get_new_case_attacks_mask(input, X_train_tensor))

        input = input.unsqueeze(0)
        # forward + backward + optimize
        outputs = model(input, new_case_attacks)
        # print(outputs[-1][-1:])
        # print(labels)
        loss = criterion(outputs[-1][-1:], labels)
        loss.backward()
        optimizer.step()

        # print statistics
        running_loss += loss.item()

   
    print('[%d, %5d] loss: %.3f' %
            (epoch + 1, i + 1, running_loss / len(X_val)))
    running_loss = 0.0

print('Finished Training')

In [ ]:
print(model.W)